In [1]:
from openai import AzureOpenAI
import httpx, requests
import json
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

from langchain_openai import AzureChatOpenAI



In [2]:
from pydantic import BaseModel, Field
from typing import List, Dict, Any
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate

In [10]:
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
model = AzureChatOpenAI(
    api_version="2024-12-01-preview",
    model="gpt-4.1-mini"
)

In [13]:
model = AzureChatOpenAI(api_version="2024-12-01-preview",model='telcogpt')
class MeetingNotes(BaseModel):
    """ Pydantic class to parse stock news description """
    summary: str = Field(description="Summary of the meeting's main discussion points")
    action_items: List[str] = Field(description="List of all action items mentioned each prefaced with dash '-'")
   

prompt_template = """
    You are a meeting assistant .
    1. Summarize the following meeting transcript in exactly a two-sentences.
    2. Then list of all action items mentionedh as a separate bullet beginning with a dash.
    
    return the result strictly as JSON with keys "summary" and "action_items"

    Transcript:
    {transcript}
    """


def getFormattedResponse(transcript: str):

    pydanticparser = PydanticOutputParser(pydantic_object=MeetingNotes)    

    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=["transcript"],
        
    )
    
    llm_input = prompt.format_prompt(transcript=transcript)
    output = model.invoke(llm_input)
    #output = model(llm_input.to_messages())

    try:
        
        output = model(llm_input.to_messages())
        if hasattr(output, 'content'):
            parsed_output = pydanticparser.parse(output.content)            
            return json.loads(parsed_output.model_dump_json())
            
        else:
            print(f"Error: Unexpected output format. No 'content' attribute found in the output: {output}")
            return None


    except Exception as e:
        print(f"An error occurred: {e}")
        print("Please ensure your language model is generating output that conforms to the specified format.")
        print(f"Format Instructions: {pydanticparser.get_format_instructions()}")
        return None
    


In [14]:


class MeetingNotes(BaseModel):
    """Pydantic class to parse meeting notes."""
    summary: str = Field(description="Summary of the meeting's main discussion points")
    action_items: List[str] = Field(description="List of all action items mentioned, each prefaced with dash '-'")

prompt_template = """
You are a meeting assistant.
1. Summarize the following meeting transcript in exactly two sentences.
2. Then list all action items mentioned, each as a separate bullet beginning with a dash.

Return the result strictly as JSON with keys "summary" and "action_items".

Transcript:
{transcript}
"""

def getFormattedResponse(transcript: str):
    pydanticparser = PydanticOutputParser(pydantic_object=MeetingNotes)

    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=["transcript"],
    )

    llm_input = prompt.format_prompt(transcript=transcript)
    output = model.invoke(llm_input)

    try:
        # Check if output is a message and has content
        if hasattr(output, 'content'):
            # Try to parse as JSON first (since prompt asks for JSON)
            try:
                output_json = json.loads(output.content)
                return output_json
            except json.JSONDecodeError:
                # Fallback: use Pydantic parser if output is not JSON
                parsed_output = pydanticparser.parse(output.content)
                return json.loads(parsed_output.model_dump_json())
        else:
            print(f"Error: Unexpected output format. No 'content' attribute found in the output: {output}")
            return None

    except Exception as e:
        print(f"An error occurred: {e}")
        print("Please ensure your language model is generating output that conforms to the specified format.")
        print(f"Format Instructions: {pydanticparser.get_format_instructions()}")
        return None



In [15]:

meeting_transcript = """ Meeting of CTAS County Commission-Transcript of Dialogue

Chairman Wormsley (at the proper time and place, after taking the chair and striking the gavel on the table): This meeting of the CTAS County Commission will come to order. Clerk please call the role. (Ensure that a majority of the members are present.)

Chairman Wormsley: Each of you has received the agenda. I will entertain a motion that the agenda be approved.

Commissioner Brown: So moved.

Commissioner Hobbs: Seconded

Chairman Wormsley: It has been moved and seconded that the agenda be approved as received by the members. All those in favor signify by saying "Aye"?...Opposed by saying "No"?...The agenda is approved. You have received a copy of the minutes of the last meeting. Are there any corrections or additions to the meeting?

Commissioner McCroskey: Mister Chairman, my name has been omitted from the Special Committee on Indigent Care.

Chairman Wormsley: Thank you. If there are no objections, the minutes will be corrected to include the name of Commissioner McCroskey. Will the clerk please make this correction. Any further corrections? Seeing none, without objection the minutes will stand approved as read. (This is sort of a short cut way that is commonly used for approval of minutes and/or the agenda rather than requiring a motion and second.)

Chairman Wormsley: Commissioner Adkins, the first item on the agenda is yours.
"""


In [16]:
parsed_op = getFormattedResponse(meeting_transcript)

In [17]:
parsed_op

{'summary': "The CTAS County Commission meeting commenced with the approval of the agenda and the correction of the minutes to include Commissioner McCroskey's name in the Special Committee on Indigent Care. The chairman then handed over the floor to Commissioner Adkins to present the first agenda item.",
 'action_items': ["- Clerk to correct the minutes to include Commissioner McCroskey's name in the Special Committee on Indigent Care."]}